# Снижение размерности: карта многомерных данных

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Снижение размерности».

## Что делает метод

Снижение размерности помогает увидеть данные с десятками или тысячами признаков. Человек не способен представить многомерное пространство, но алгоритм сжимает его до двух-трёх осей так, чтобы сохранить главное — взаимное расположение наблюдений. На полученной карте сразу видно, образуют ли данные группы, есть ли выбросы и плавные переходы.

В этом блокноте мы применим два метода — линейный (метод главных компонент, PCA) и нелинейный (UMAP с запасным вариантом t-SNE), — сравним их карты и разберём, как их корректно читать. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте географическую карту: трёхмерная поверхность Земли разворачивается в плоский лист. Часть информации неизбежно теряется (горы кажутся плоскими), но взаимное расположение городов остаётся читаемым. Снижение размерности делает то же самое с многомерными научными данными: сжимает десятки или тысячи признаков до двух-трёх осей, сохраняя главное — взаимное расположение наблюдений.

На полученной карте сразу видно: образуют ли пробы группы, есть ли выбросы, плавно ли переходят условия эксперимента. Это инструмент разведки — он строит гипотезы, а не подтверждает их.

**Ключевые понятия, которые встретятся в блокноте:**

- **Признак** — один измеренный параметр наблюдения (столбец таблицы).
- **Размерность** — число признаков; «высокоразмерные данные» — таблица с очень многими столбцами.
- **Главная компонента (PCA)** — новая ось, построенная так, чтобы вдоль неё разброс данных был максимальным; первая компонента объясняет больше всего вариации, вторая — следующую по величине.
- **Доля объяснённой дисперсии** — какой процент общего разброса данных «видно» через выбранные компоненты.
- **Линейный метод (PCA)** — новые оси строятся как взвешенные суммы исходных признаков; расстояния на карте приблизительно отражают исходные.
- **Нелинейный метод (t-SNE, UMAP)** — оси строятся иначе: упор на сохранение локального соседства, а не глобальных расстояний. Кластеры выделяются чётче, но абсолютные расстояния между ними и их размеры интерпретировать нельзя.
- **Стандартизация** — приведение признаков к единому масштабу перед снижением размерности; обязательный шаг.

## 1. Установка библиотек

PCA и t-SNE входят в `scikit-learn` и не требуют сторонних пакетов. UMAP — родственный нелинейный метод; он устанавливается отдельным пакетом `umap-learn` и в этом блокноте предлагается как необязательная альтернатива (см. раздел 6).

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор рукописных цифр (`digits` из `scikit-learn`): 1797 изображений 8×8 пикселей, то есть **64 признака** на каждое наблюдение (яркости 64 пикселей). Это удобная модель многомерных данных: глазами 64-мерное пространство не охватить, а карта снижения размерности покажет, группируются ли изображения по цифрам.

Класс цифры (`y`) используется только для раскраски карты — в самом построении проекции он не участвует. Ячейка ниже загружает данные.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits

data = load_digits()
X = data.data            # 64 признака (яркости пикселей)
y = data.target          # класс цифры, только для раскраски

print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}")

## 4. Применение метода

**Шаг 4.1. Стандартизация.**

Перед снижением размерности признаки стандартизуем — чтобы признаки с большим числовым масштабом не доминировали над остальными.

**Шаг 4.2. Линейная проекция — метод главных компонент (PCA).**

PCA находит новые оси — **главные компоненты** — упорядоченные по убыванию объяснённой дисперсии. Первая ось направлена вдоль максимального разброса данных, вторая — вдоль следующего по величине и перпендикулярна первой. Мы берём только две компоненты, чтобы отобразить данные на плоскости.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_scaled = StandardScaler().fit_transform(X)

# Линейная проекция: метод главных компонент.
pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_
print(f"Доля объяснённой дисперсии по двум компонентам: "
      f"{explained[0]:.2%} и {explained[1]:.2%}")

**Шаг 4.3. Нелинейная проекция — t-SNE.**

t-SNE (t-distributed Stochastic Neighbor Embedding) работает иначе: он старается сохранить **локальное соседство** — наблюдения, похожие в исходном пространстве, окажутся рядом на карте. Кластеры на карте t-SNE обычно выделены чётче, чем на PCA, но расстояния и размеры кластеров интерпретировать буквально нельзя — они не отражают исходные.

Параметр `perplexity=30` — гиперпараметр t-SNE: примерно отвечает за то, сколько соседей учитывается при построении проекции (аналог «радиуса обзора»).

In [ ]:
# Нелинейная проекция: t-SNE (входит в scikit-learn, дополнительных пакетов не требует).
from sklearn.manifold import TSNE

coords_nl = TSNE(n_components=2, perplexity=30,
                 init="pca", random_state=42).fit_transform(X_scaled)
nl_name = "t-SNE"
print(f"Нелинейная проекция построена методом {nl_name}.")

### Шаг 4.4. Сравнение проекций

Ячейка ниже строит две карты рядом. Цвет каждой точки — цифра, которую изображает наблюдение (0–9). Так видно, насколько каждый метод «разводит» разные цифры в разные части карты.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6), constrained_layout=True)

for ax, coords, title in [
    (axes[0], coords_pca, "PCA (линейная проекция)"),
    (axes[1], coords_nl, f"{nl_name} (нелинейная проекция)"),
]:
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=y, cmap="tab10",
                    s=14, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel("Ось 1")
    ax.set_ylabel("Ось 2")

cbar = fig.colorbar(sc, ax=axes, fraction=0.04, pad=0.03,
                    ticks=range(10))
cbar.set_label("Класс цифры")
plt.show()

**Как читать эти карты:**

- **PCA (левый)**: оси имеют физический смысл — вдоль каждой меняется определённая комбинация исходных признаков. Расстояния между точками приблизительно отражают исходные. Цифры частично перекрываются — две компоненты объясняют лишь часть дисперсии (см. напечатанные доли выше).

- **t-SNE (правый)**: кластеры выделены чётче, похожие цифры сгруппированы. Однако оси не имеют интерпретации, а расстояния между кластерами и их размеры не несут количественного смысла. Главное на такой карте — факт разделения или слияния групп и кто с кем соседствует.

Цветовая шкала справа — номер цифры (0–9).

## 5. Интерпретация результата

- **PCA** даёт интерпретируемые оси, упорядоченные по доле объяснённой дисперсии; расстояния на ней приближённо отражают исходные. Если двух компонент мало, постройте график накопленной дисперсии и возьмите больше осей.
- **t-SNE** обычно нагляднее разделяет группы, но расстояния между кластерами и их размеры на такой карте интерпретировать буквально нельзя — важен лишь сам факт разделения и соседства. То же относится к UMAP — родственному нелинейному методу.
- Результат нелинейных методов зависит от параметров (`perplexity` для t-SNE, `n_neighbors` для UMAP) и инициализации — проверяйте устойчивость карты при их изменении.
- Снижение размерности всегда теряет часть информации; число осей выбирают по доле объяснённой дисперсии или по задаче.

Карта снижения размерности — инструмент разведки: она подсказывает гипотезы о структуре данных, но не заменяет количественный анализ.

## 5б. Наглядный эксперимент: сколько компонент нужно?

График накопленной объяснённой дисперсии показывает, сколько информации удерживается при разном числе компонент PCA. По нему выбирают «колено» — число компонент, после которого прирост информации становится незначительным.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Полный PCA для анализа дисперсии.
pca_full = PCA(random_state=42).fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

# Находим число компонент для 80 % и 95 % дисперсии.
n80 = int(np.searchsorted(cumvar, 0.80)) + 1
n95 = int(np.searchsorted(cumvar, 0.95)) + 1

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar * 100,
        color=VIZ["series"][0], linewidth=2)
ax.axhline(80, color=VIZ["series"][2], linestyle="--", linewidth=1.2,
           label=f"80 % дисперсии — {n80} компонент")
ax.axhline(95, color=VIZ["series"][1], linestyle="--", linewidth=1.2,
           label=f"95 % дисперсии — {n95} компонент")
ax.set_title("Накопленная объяснённая дисперсия PCA\n(набор рукописных цифр, 64 признака)")
ax.set_xlabel("Число главных компонент")
ax.set_ylabel("Накопленная дисперсия, %")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()
print(f"2 компоненты объясняют {cumvar[1]*100:.1f}% дисперсии")

**Как читать этот график:**

Ось X — число взятых главных компонент. Ось Y — накопленный процент дисперсии, который они объясняют. Кривая должна резко подниматься вначале и выходить на насыщение.

- Пунктирная оранжевая линия показывает, сколько компонент нужно, чтобы объяснить 80 % дисперсии.
- Пунктирная зелёная — для 95 %.

Именно поэтому 2 компоненты (которые мы взяли для карт выше) объясняют лишь часть информации: цифры частично перекрываются на карте PCA. Это нормально — для полноценного анализа нужно больше компонент.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу числовых признаков; столбец с метками не обязателен — он нужен лишь для раскраски карты.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже, укажите имя файла и при наличии — столбец с метками.
3. Последовательно выполните ячейки разделов 4 и 5.

Чтобы дополнительно построить проекцию UMAP, установите пакет командой `%pip install -q umap-learn` и выполните ячейку ниже — она задаёт `coords_nl` методом UMAP вместо t-SNE.

## Поэкспериментируйте сами

На основных картах (раздел 4) попробуйте изменить параметры и перестроить графики:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `perplexity` в t-SNE | Попробуйте 5, 50, 100 | Как меняется размер и связность кластеров? |
| `n_components=3` в PCA | Добавьте третью компоненту | Сколько дисперсии объясняется тремя осями? |
| `n_components=2` при `PCA(n_components=0.95)` | Замените: число компонент, объясняющих 95 % дисперсии | Сколько компонент выбирает алгоритм автоматически? |

Попробуйте закрасить точки не по цифре, а по любому другому признаку — например, по первой главной компоненте.

## Краткие выводы

- Снижение размерности — инструмент разведки: оно строит гипотезы о структуре данных, но не заменяет количественный анализ.
- PCA: интерпретируемые оси, сохраняет приблизительные расстояния, быстро работает. Выбирайте число компонент по доле объяснённой дисперсии.
- t-SNE и UMAP: нагляднее выделяют локальные кластеры, но расстояния между кластерами и их размеры интерпретировать буквально нельзя. Результат зависит от гиперпараметров — проверяйте устойчивость.
- Стандартизация обязательна перед PCA; при наличии признаков с разными единицами без неё компоненты бессмысленны.
- На практике: сначала PCA (понять структуру, выбрать число компонент), затем t-SNE/UMAP поверх PCA (для финальной визуализации).

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На карте PCA цифры 4 и 9 частично перекрываются, тогда как на карте t-SNE они образуют отдельные плотные кластеры. При этом две главные компоненты объясняют лишь 29 % дисперсии. Как правильно интерпретировать каждый из этих фактов и что они совместно говорят о структуре данных?
2. Коллега перезапустил t-SNE с `perplexity=100` и получил карту, на которой почти все цифры слились в одно облако, тогда как при `perplexity=30` они были разделены. Означает ли это, что разделение при `perplexity=30` было артефактом? Как проверить устойчивость результата?
3. Вы хотите использовать координаты t-SNE как признаки для последующей классификации. Назовите две принципиальные причины, по которым это некорректно, и предложите правильную альтернативу.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Перекрытие на PCA при 29 % объяснённой дисперсии означает, что два компонента не передают достаточно информации для линейного разделения — это ограничение метода, а не отсутствие структуры. Чёткое разделение на t-SNE указывает, что структура есть, но она нелинейная. Совместный вывод: данные имеют выраженную кластерную структуру, которая не видна в двух главных компонентах, но присутствует в высокоразмерном пространстве.
2. Нет, это не обязательно артефакт. t-SNE чувствителен к `perplexity`: большие значения усредняют локальную структуру, и кластеры могут сливаться. Устойчивость проверяется: (а) несколькими запусками при одном `perplexity` с разными `random_state` — карты должны быть качественно похожими; (б) UMAP с разными `n_neighbors` — независимый нелинейный метод; (в) численными метриками — силуэт в исходном 64-мерном пространстве, а не на карте.
3. Первая причина: t-SNE не имеет стандартного механизма проецирования новых точек — координаты для тестовых данных нельзя получить из обучающей проекции. Вторая причина: расстояния и масштаб на карте t-SNE не несут количественного смысла, поэтому координаты на ней не являются информативными числовыми признаками. Правильная альтернатива — использовать главные компоненты PCA (в нужном числе для 95 % дисперсии) или исходные признаки со снижением размерности через `PCA.transform()`.
</details>


In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv("my_data.csv")               # ваш файл
# label_column = None                           # имя столбца с метками или None
#
# if label_column is not None:
#     y = df[label_column].astype("category").cat.codes.to_numpy()
#     X = df.drop(columns=[label_column])
# else:
#     y = np.zeros(len(df), dtype=int)
#     X = df.copy()
# X = X.select_dtypes(include="number").to_numpy()
#
# print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}")
# Далее повторите ячейки раздела 4.

In [ ]:
# --- Необязательно: проекция UMAP вместо t-SNE ---
# Выполните после установки пакета: %pip install -q umap-learn
# import umap
# reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
# coords_nl = reducer.fit_transform(X_scaled)
# nl_name = "UMAP"
# print(f"Нелинейная проекция построена методом {nl_name}.")
# После этого повторно выполните ячейку с графиком из раздела 4.